In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv(r"E:\vs\base_module1\Insurance Premium Prediction\dataset\train.csv")

df.head()


,id,Age,Gender,Annual Income,Marital Status,Number of Dependents,Education Level,Occupation,Health Score,Location,...,Previous Claims,Vehicle Age,Credit Score,Insurance Duration,Policy Start Date,Customer Feedback,Smoking Status,Exercise Frequency,Property Type,Premium Amount
0,0,19.0,Female,10049.0,Married,1.0,Bachelor's,Self-Employed,22.598761,Urban,...,2.0,17.0,372.0,5.0,2023-12-23 15:21:39.134960,Poor,No,Weekly,House,2869.0
1,1,39.0,Female,31678.0,Divorced,3.0,Master's,NaN,15.569731,Rural,...,1.0,12.0,694.0,2.0,2023-06-12 15:21:39.111551,Average,Yes,Monthly,House,1483.0
2,2,23.0,Male,25602.0,Divorced,3.0,High School,Self-Employed,47.177549,Suburban,...,1.0,14.0,NaN,3.0,2023-09-30 15:21:39.221386,Good,Yes,Weekly,House,567.0
3,3,21.0,Male,141855.0,Married,2.0,Bachelor's,NaN,10.938144,Rural,...,1.0,0.0,367.0,1.0,2024-06-12 15:21:39.226954,Poor,Yes,Daily,Apartment,765.0
4,4,21.0,Male,39651.0,Single,1.0,Bachelor's,Self-Employed,20.376094,Rural,...,0.0,8.0,598.0,4.0,2021-12-01 15:21:39.252145,Poor,Yes,Weekly,House,2022.0


In [5]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print(df.columns.tolist())
df.info()


Rows: 1200000
Columns: 21
['id', 'Age', 'Gender', 'Annual Income', 'Marital Status', 'Number of Dependents', 'Education Level', 'Occupation', 'Health Score', 'Location', 'Policy Type', 'Previous Claims', 'Vehicle Age', 'Credit Score', 'Insurance Duration', 'Policy Start Date', 'Customer Feedback', 'Smoking Status', 'Exercise Frequency', 'Property Type', 'Premium Amount']
<class 'pandas.DataFrame'>
RangeIndex: 1200000 entries, 0 to 1199999
Data columns (total 21 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   id                    1200000 non-null  int64  
 1   Age                   1181295 non-null  float64
 2   Gender                1200000 non-null  str    
 3   Annual Income         1155051 non-null  float64
 4   Marital Status        1181471 non-null  str    
 5   Number of Dependents  1090328 non-null  float64
 6   Education Level       1200000 non-null  str    
 7   Occupation            841925 non-null   st

In [6]:
df.isnull().sum()

id                           0
Age                      18705
Gender                       0
Annual Income            44949
Marital Status           18529
Number of Dependents    109672
Education Level              0
Occupation              358075
Health Score             74076
Location                     0
Policy Type                  0
Previous Claims         364029
Vehicle Age                  6
Credit Score            137882
Insurance Duration           1
Policy Start Date            0
Customer Feedback        77824
Smoking Status               0
Exercise Frequency           0
Property Type                0
Premium Amount               0
dtype: int64

In [7]:
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_table = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": missing_percent
})

missing_table.sort_values("Missing Percentage", ascending=False)

,Missing Count,Missing Percentage
Previous Claims,364029,30.335750
Occupation,358075,29.839583
Credit Score,137882,11.490167
Number of Dependents,109672,9.139333
Customer Feedback,77824,6.485333
Health Score,74076,6.173000
Annual Income,44949,3.745750
Age,18705,1.558750
Marital Status,18529,1.544083
Vehicle Age,6,0.000500


In [8]:
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

print("Numerical Columns:")
print(numerical_cols)
categorical_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
print("Categorical Columns:")
print(categorical_cols)
X = df.drop("Premium Amount", axis=1)
y = df["Premium Amount"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Numerical Columns:
['id', 'Age', 'Annual Income', 'Number of Dependents', 'Health Score', 'Previous Claims', 'Vehicle Age', 'Credit Score', 'Insurance Duration', 'Premium Amount']
Categorical Columns:
['Gender', 'Marital Status', 'Education Level', 'Occupation', 'Location', 'Policy Type', 'Policy Start Date', 'Customer Feedback', 'Smoking Status', 'Exercise Frequency', 'Property Type']
Features shape: (1200000, 20)
Target shape: (1200000,)


In [9]:
# Reset X and y from original dataframe

X = df.drop("Premium Amount", axis=1).copy()
y = df["Premium Amount"].copy()

# 1. Remove ID
X = X.drop("id", axis=1)

# 2. Convert date column
X["Policy Start Date"] = pd.to_datetime(X["Policy Start Date"])

# 3. Extract date features
X["Policy Start Year"] = X["Policy Start Date"].dt.year
X["Policy Start Month"] = X["Policy Start Date"].dt.month
X["Policy Start Day"] = X["Policy Start Date"].dt.day

# Remove original date column
X = X.drop("Policy Start Date", axis=1)

# 4. Identify numerical and categorical columns
numerical_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "str"]).columns.tolist()

# 5. Fill numerical missing values with median
for col in numerical_features:
    X[col] = X[col].fillna(X[col].median())

# 6. Fill categorical missing values with mode
for col in categorical_features:
    X[col] = X[col].fillna(X[col].mode()[0])

# 7. One-hot encoding
X = pd.get_dummies(
    X,
    columns=categorical_features,
    drop_first=True,
    dtype=int
)

print("Final X shape:", X.shape)
print("Target shape:", y.shape)
print("Missing values:", X.isnull().sum().sum())

Final X shape: (1200000, 31)
Target shape: (1200000,)
Missing values: 0


In [10]:
from sklearn.model_selection import train_test_split

# First split: 60% Train, 40% Temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=42
)

# Second split: 20% Validation, 20% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set:", X_test.shape, y_test.shape)

Training set: (720000, 31) (720000,)
Validation set: (240000, 31) (240000,)
Test set: (240000, 31) (240000,)


In [11]:
total = len(df)

print("Train:", len(X_train) / total * 100, "%")
print("Validation:", len(X_val) / total * 100, "%")
print("Test:", len(X_test) / total * 100, "%")

Train: 60.0 %
Validation: 20.0 %
Test: 20.0 %


In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform validation and test data
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaling completed!")

Scaling completed!


In [13]:
print("X_train_scaled:", X_train_scaled.shape)
print("X_val_scaled:", X_val_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)

X_train_scaled: (720000, 31)
X_val_scaled: (240000, 31)
X_test_scaled: (240000, 31)


In [14]:
X_train_b = np.c_[np.ones(X_train_scaled.shape[0]), X_train_scaled]
X_val_b = np.c_[np.ones(X_val_scaled.shape[0]), X_val_scaled]
X_test_b = np.c_[np.ones(X_test_scaled.shape[0]), X_test_scaled]

print("Training shape:", X_train_b.shape)
print("Validation shape:", X_val_b.shape)
print("Test shape:", X_test_b.shape)

Training shape: (720000, 32)
Validation shape: (240000, 32)
Test shape: (240000, 32)


In [15]:
def ridge_normal_equation(X, y, lambda_value):
    n_features = X.shape[1]

    I = np.eye(n_features)

    I[0, 0] = 0

    weights = np.linalg.inv(
        X.T @ X + lambda_value * I
    ) @ X.T @ y

    return weights

In [16]:
lambda_value = 1

weights = ridge_normal_equation(
    X_train_b,
    y_train.values,
    lambda_value
)

print("Number of weights:", len(weights))

Number of weights: 32


In [17]:
y_val_pred = X_val_b @ weights

print(y_val_pred[:10])

[1072.62905305 1044.83951652 1070.51950191 1259.59471718 1099.94957605
 1080.67976583 1067.23913687 1126.08514329 1084.98173754 1129.3919523 ]


In [18]:
from sklearn.metrics import mean_squared_error, r2_score

mse = mean_squared_error(y_val, y_val_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_val, y_val_pred)

print("MSE:", mse)
print("RMSE:", rmse)
print("R²:", r2)

MSE: 745286.2795825677
RMSE: 863.2996464626681
R²: 0.0028280868783372437


In [19]:
lambda_values = [0, 0.1, 1, 10, 100]

results = []

for lambda_value in lambda_values:

    weights = ridge_normal_equation(
        X_train_b,
        y_train.values,
        lambda_value
    )

    y_val_pred = X_val_b @ weights

    mse = mean_squared_error(y_val, y_val_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_val, y_val_pred)

    results.append([
        lambda_value,
        mse,
        rmse,
        r2
    ])
    results_df = pd.DataFrame(
    results,
    columns=["Lambda", "MSE", "RMSE", "R2"]
)

results_df

,Lambda,MSE,RMSE,R2
0,0.0,745286.279791,863.299647,0.002828
1,0.1,745286.279770,863.299647,0.002828
2,1.0,745286.279583,863.299646,0.002828
3,10.0,745286.277704,863.299645,0.002828
4,100.0,745286.258977,863.299635,0.002828


In [20]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

lambda_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000]

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = []

for lambda_value in lambda_values:

    fold_mse = []

    for train_index, val_index in kf.split(X_train_b):

        X_fold_train = X_train_b[train_index]
        X_fold_val = X_train_b[val_index]

        y_fold_train = y_train.values[train_index]
        y_fold_val = y_train.values[val_index]

        weights = ridge_normal_equation(
            X_fold_train,
            y_fold_train,
            lambda_value
        )

        y_fold_pred = X_fold_val @ weights

        mse = mean_squared_error(
            y_fold_val,
            y_fold_pred
        )

        fold_mse.append(mse)

    avg_mse = np.mean(fold_mse)

    cv_results.append([
        lambda_value,
        avg_mse
    ])

cv_results_df = pd.DataFrame(
    cv_results,
    columns=["Lambda", "Average MSE"]
)

cv_results_df = cv_results_df.sort_values(
    "Average MSE"
)

cv_results_df

,Lambda,Average MSE
6,1000.000,747087.404460
5,100.000,747087.559849
4,10.000,747087.576221
3,1.000,747087.577866
2,0.100,747087.578031
1,0.010,747087.578047
0,0.001,747087.578049


In [21]:
best_lambda = cv_results_df.iloc[0]["Lambda"]

print("Best Lambda:", best_lambda)

Best Lambda: 1000.0


In [22]:
best_lambda = 1000

final_weights = ridge_normal_equation(
    X_train_b,
    y_train.values,
    best_lambda
)

y_test_pred = X_test_b @ final_weights

test_mse = mean_squared_error(
    y_test,
    y_test_pred
)

test_rmse = np.sqrt(test_mse)

test_r2 = r2_score(
    y_test,
    y_test_pred
)

print("Best Lambda:", best_lambda)
print("Test MSE:", test_mse)
print("Test RMSE:", test_rmse)
print("Test R²:", test_r2)

Best Lambda: 1000
Test MSE: 743901.4699109244
Test RMSE: 862.4972289294177
Test R²: 0.0027719963183544527


In [23]:
import pickle
 
best_lambda = 1000

final_weights = ridge_normal_equation(
    X_train_b,
    y_train.values,
    best_lambda
)

model_data = {
    "weights": final_weights,
    "scaler": scaler,
    "feature_columns": X_train.columns.tolist()
}

with open("insurance_ridge_model.pkl", "wb") as f:
    pickle.dump(model_data, f)

print("Model saved successfully!")

Model saved successfully!
